# 03 – Exploratory Analysis & Feature Selection
**Owner:** Jinash  
**Goal:** Explore relationships with winning, measure home-court advantage, and apply two feature selection methods.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
FIGURES = Path("../reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

In [ ]:
df = pd.read_csv(PROCESSED / "games_processed.csv")
features = [c for c in df.columns if c.endswith("_diff")]
X, y = df[features], df["HOME_TEAM_WINS"]

## Home-court advantage by season

In [ ]:
season_rate = df.groupby("SEASON")["HOME_TEAM_WINS"].mean()
ax = season_rate.plot(marker="o", color="black")
ax.axhline(0.5, linestyle="--", color="gray")
ax.set(title="Home win rate by season", ylabel="Home win rate")
plt.savefig(FIGURES / "home_win_rate_by_season.png", bbox_inches="tight")
plt.show()

## Distributions by outcome

In [ ]:
fig, axes = plt.subplots(1, len(features), figsize=(4 * len(features), 3.5))
for ax, f in zip(axes, features):
    sns.kdeplot(data=df, x=f, hue="HOME_TEAM_WINS", ax=ax, common_norm=False)
plt.tight_layout()
plt.show()

## Method 1: Correlation with the target

In [ ]:
corr = df[features + ["HOME_TEAM_WINS"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Greys")
plt.show()
corr["HOME_TEAM_WINS"].drop("HOME_TEAM_WINS").sort_values(ascending=False)

## Method 2: Random forest feature importance

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(X, y)
rf_imp = pd.Series(rf.feature_importances_, index=features).sort_values()
rf_imp.plot(kind="barh", color="black", title="Random forest importance")
plt.show()

## (Optional) Method 3: Mutual information

In [ ]:
mi = pd.Series(mutual_info_classif(X, y, random_state=RANDOM_STATE), index=features).sort_values(ascending=False)
mi

## Final feature set
Document which features are kept and why.
- 